# CS4337 Fall 2026 — Starter Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TXSTCODEPLAYGROUND/CodeCS4337Fall2026/blob/main/starternotebook.ipynb)

Run the cells from top to bottom.

1. **Use a GPU:** *Runtime → Change runtime type → GPU* (training also works on CPU, just slower).
2. **Mount Google Drive** so your results survive runtime resets.
3. **Set up the repo:** clone (or update) it, apply the Colab settings, and install only the missing requirements.
4. **Run a project**, either as a terminal command or from Python.

Before a long training run, read the [pro tips](#pro-tips-keep-training-running) at the end.

To run a different project, change `PROJECT` in the setup cell and the project name in the run cells.

## 1. Setup

Run these three cells once per session (and again after the runtime resets). The setup cell is safe to re-run, e.g. to get new projects with `git pull`.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
%%bash
set -e
PROJECT=TrainingBasicConvnet

cd /content
if [ -d CodeCS4337Fall2026/.git ]; then
  git -C CodeCS4337Fall2026 pull --ff-only
else
  git clone https://github.com/TXSTCODEPLAYGROUND/CodeCS4337Fall2026.git
fi
cd CodeCS4337Fall2026

# Colab settings: data on the local disk, results in Google Drive.
cp .envcolab .env

# Install only packages Colab does not already have (versions are not pinned).
grep -vE '^\s*(#|$)' "$PROJECT/requirements.txt" \
  | sed -E 's/[<>=!~;[ ].*//' \
  | while read -r pkg; do
      if pip show "$pkg" > /dev/null 2>&1; then
        echo "skip $pkg (already installed)"
      else
        pip install -q "$pkg" && echo "installed $pkg"
      fi
    done

In [ ]:
# The bash cell above cannot change the notebook's working directory, so do it here.
%cd /content/CodeCS4337Fall2026

## 2. Run a project (choose one option)

Both options run exactly the same training and save results to the same Drive folder. Pass a config name from the project's `configs/` folder. To try your own settings, copy a config in the file browser (left sidebar), edit it, and pass its name.

| Option | How | When to use it |
| --- | --- | --- |
| **A. Terminal command** | `!` in a notebook cell, or Colab's terminal (Colab Pro) | Same command as on your own machine |
| **B. From Python** | `from TrainingBasicConvnet import main` | Simple, everything stays in the notebook |

### Option A: Run as a terminal command

**A1. From a notebook cell (everyone):** a line starting with `!` is sent to the shell. Run the cell below.

In [ ]:
!cd /content/CodeCS4337Fall2026 && python -m TrainingBasicConvnet --config config01

**A2. From Colab's terminal (Colab Pro):** open the terminal from the **Terminal** button at the bottom left of the Colab window, then run:

```bash
cd /content/CodeCS4337Fall2026
python -m TrainingBasicConvnet --config config01
```

or, from inside the project folder:

```bash
cd /content/CodeCS4337Fall2026/TrainingBasicConvnet
python main.py --config config01
```

The terminal runs on the same machine as this notebook, so it uses the same files, packages, and Drive mount. **You still need to run the setup cells above first**: Drive can only be mounted from a notebook cell. While training runs in the terminal, the notebook stays free for other work.

### Option B: Run from Python

This cell adds the repo to Python's import path, so it works even if the directory-change cell in setup was skipped. Option A does not need that cell either: its command changes directory on its own.

In [ ]:
import sys

repo = "/content/CodeCS4337Fall2026"
if repo not in sys.path:
    sys.path.insert(0, repo)

from TrainingBasicConvnet import main

main("config01")

## 3. Results

Each run is saved in Google Drive under `MyDrive/CodeCS4337Fall2026/runs/<Project>/<config>/<timestamp>/`. `runs_summary.csv` has one row per run.

In [ ]:
!cat /content/drive/MyDrive/CodeCS4337Fall2026/runs/TrainingBasicConvnet/config01/runs_summary.csv

## Pro tips: keep training running

Colab disconnects runtimes that look idle (no interaction for a while, often around 90 minutes) and also caps how long a runtime can run in total (around 12 hours on the free tier, longer on paid plans). The exact limits vary and are not guaranteed. When the runtime disconnects, training stops and everything under `/content/` (outside Drive) is deleted.

**Save your state in Google Drive**
- Always mount Drive *before* training. Results then go straight to Drive (`OUTPUT_DIR` in `.envcolab`), not to the temporary disk.
- The best model so far (`best_epochXX_valaccY.pt`) is saved to Drive *during* training, every time validation accuracy improves. If the runtime disconnects mid-run, that checkpoint is kept.
- The last checkpoint, `history.json`, the results file, and the row in `runs_summary.csv` are written only when the run finishes, so an interrupted run has only its best checkpoint.
- Keep your notebook in Drive too: *File → Save a copy in Drive*. Keep copies of configs you edited in Drive, since `/content/CodeCS4337Fall2026` is deleted on reset.

**Avoid idle disconnects**
- Keep the Colab browser tab open, and don't let your computer go to sleep (plug in your laptop and turn off sleep while training).
- Check on the notebook now and then (scroll, look at the progress bars). Running in the terminal does *not* prevent disconnects: the runtime is shared.
- Don't use scripts or browser extensions that fake activity (auto-clickers); they can get your Colab usage restricted.
- With Colab Pro+, *background execution* keeps the runtime alive even after you close the browser.

**Plan your runs**
- Test a new config with few epochs first (e.g. `"epochs": 1`), then start the full run.
- Prefer several shorter runs over one very long run; each run saves its own results.
- After a disconnect, re-run the setup cells, then start the run again.
- When you are done, free the GPU with *Runtime → Disconnect and delete runtime*. It saves your GPU quota (or compute units on paid plans) for next time.